# 02 · Train / fine-tune GTCRN on Colab (backup to the Kaggle notebook)
Everything important is saved to Google Drive and training **resumes automatically** if you re-run the cells. Datasets are downloaded to the local disk each session and are not kept.
Put `voice-enhancer-m3c.zip` (or newer) in Drive / `voice-enhancer/`, then Runtime > Change runtime type > GPU. If Colab says no GPU is available, wait (free quota) or use the Kaggle notebook.
Run the cells in order. If one fails, copy me the error text.

In [ ]:
import os, glob, shutil, subprocess, time, torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/voice-enhancer"; os.makedirs(DRIVE, exist_ok=True)
zips = sorted(glob.glob(DRIVE + "/voice-enhancer-m3*.zip"))
assert zips, f"Upload the voice-enhancer-m3 zip into {DRIVE} first"
!unzip -q -o "{zips[-1]}" 'voice-enhancer/ai-training/*' -d /content/
%cd /content/voice-enhancer/ai-training
!pip -q install soundfile tensorboard
subprocess.run("apt-get -qq update && apt-get -qq install -y aria2 > /dev/null", shell=True)
print("aria2c:", shutil.which("aria2c"))
if not os.path.exists("/content/gtcrn"):
    !git clone -q https://github.com/Xiaobin-Rong/gtcrn /content/gtcrn
CKPT = (glob.glob("/content/gtcrn/checkpoints/*dns3*") or [None])[0]
print("pretrained checkpoint:", CKPT)

In [ ]:
!python smoke_test.py --gtcrn-dir /content/gtcrn

In [ ]:
DATA = "/content"
t = time.time()
!python fetch_datasets.py fetch libritts_r_train_clean_100 libritts_r_dev_clean libritts_r_test_clean rirs_noises --root {DATA}/datasets --delete-archive
print(f"download + unpack: {(time.time() - t) / 60:.1f} min")
!cp {DATA}/datasets/MANIFEST.json "{DRIVE}/MANIFEST.json"

In [ ]:
MAX_FILES = 3000     # None = all files
flag = f"--max-files {MAX_FILES}" if MAX_FILES else ""
!python prepare_data.py --speech {DATA}/datasets/libritts_r_train_clean_100:train {DATA}/datasets/libritts_r_dev_clean:val {DATA}/datasets/libritts_r_test_clean:test --noise {DATA}/datasets/rirs_noises --out {DATA}/prepared {flag}
for n in ("speech_train", "speech_val", "noise_train", "noise_val", "rir_train", "rir_val"):
    k = sum(1 for l in open(f"{DATA}/prepared/lists/{n}.txt") if l.strip())
    print(f"{n:13} {k:7,d}")
    assert k > 0, f"{n} list is empty: re-run the download and prepare cells in order"

Training: watch `val SI-SDR` (should rise) and the `high-band energy` line (3-5k / 5-7k / 7-8k should stay near 0 dB; below about -6 dB means a dull voice, stop and tell me).
Stopping the cell and re-running continues from the last finished epoch.

In [ ]:
EPOCHS, STEPS, BATCH, LR = 12, 1000, 16, 2e-4
RUN = f"{DRIVE}/runs/gtcrn_ft1"
os.makedirs(RUN, exist_ok=True)
resume = f"--resume {RUN}/last.pt" if os.path.exists(f"{RUN}/last.pt") else ""
init = f"--init {CKPT}" if CKPT and not resume else ""
!python train.py --lists {DATA}/prepared/lists --arch gtcrn --gtcrn-dir /content/gtcrn {init} --preset farfield --epochs {EPOCHS} --steps-per-epoch {STEPS} --batch {BATCH} --lr {LR} --workers 2 --out "{RUN}" {resume}

## Try it on a clip (upload an original; the result downloads)

In [ ]:
from google.colab import files
up = files.upload(); clip = list(up)[0]; open(clip, "wb").write(up[clip])
out = "ft1_" + os.path.splitext(clip)[0] + ".wav"
!python infer.py --ckpt "{RUN}/best.pt" --arch gtcrn --gtcrn-dir /content/gtcrn --inp "{clip}" --out "{out}"
!python bench_clips.py --orig "{clip}" --enh "{out}"
files.download(out)